# Regressão Logística (modelo principal)

**Nota sobre o nome (feedback #7):** o problema é de **classificação binária** (`mudanca_CPAK` ∈ {0,1}). O modelo é uma **Regressão Logística** (`LogisticRegression`) — *não* uma regressão linear. Toda a designação abaixo é, portanto, "Regressão Logística".

**Objetivo:** treinar e afinar a Regressão Logística, avaliando o desempenho no conjunto de teste. O pré-processamento (imputação, one-hot encoding e scaling) é partilhado via `preprocessing.py` e ajustado apenas com o treino, dentro de uma pipeline (feedback #1, #5, #6).

## 1.1) Importação de bibliotecas e leitura dos dados

In [1]:
# importações de bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score,
)

import preprocessing as prep

# Carregamento com as transformações determinísticas; imputação/encoding/scaling
# ficam na pipeline (feedback #1, #5). Não se usa o ortho_eda_clean.csv (já imputado).
X, y = prep.carregar_dados()

## 2) Preparação dos dados

In [2]:
# Split estratificado único (feedback #1). O teste é usado uma só vez, no fim.
X_train, X_test, y_train, y_test = prep.dividir_treino_teste(X, y)

In [3]:
# Verificação da distribuição da target no conjunto de treino
print("Distribuição da Target (CPAK) - Treino")
print("N. observações Treino", len(X_train))
print(y_train.value_counts(normalize=True))

Distribuição da Target (CPAK) - Treino
N. observações Treino 183
0    0.901639
1    0.098361
Name: proportion, dtype: float64


In [4]:
# Verificação da distribuição da target no conjunto de teste

print("Distribuição da Target (CPAK) - Teste")
print("N. observações Teste", len(X_test))
print(y_test.value_counts(normalize=True))

Distribuição da Target (CPAK) - Teste
N. observações Teste 79
0    0.898734
1    0.101266
Name: proportion, dtype: float64


## 3) Baseline de Regressão Logística

In [5]:
# Baseline: Regressão Logística dentro da pipeline (imputação + OHE + scaling).
# O scaling (escalar=True) é importante para a convergência da LR.
pipe_lr_base = prep.construir_pipeline(
    LogisticRegression(max_iter=1000, random_state=42), X, escalar=True
)
pipe_lr_base.fit(X_train, y_train)

,steps,"[('prep', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [6]:
# Avaliação do baseline no teste: matriz de confusão + relatório + AUC (probabilidades)
res_lr_base = prep.avaliar_teste(pipe_lr_base, X_test, y_test, titulo="Regressão Logística baseline")

              precision    recall  f1-score   support

           0     0.9167    0.9296    0.9231        71
           1     0.2857    0.2500    0.2667         8

    accuracy                         0.8608        79
   macro avg     0.6012    0.5898    0.5949        79
weighted avg     0.8528    0.8608    0.8566        79

AUC (probabilidades): 0.7165


C:\Users\edu23\Desktop\Projetos_local\Machine_Learning\preprocessing.py:167: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Validação cruzada estratificada APENAS no treino (feedback #1), AUC de probabilidades
print("=== Validação cruzada (treino) — Regressão Logística baseline ===")
_ = prep.validacao_cruzada_treino(pipe_lr_base, X_train, y_train)

=== Validação cruzada (treino) — Regressão Logística baseline ===


              precision    recall  f1-score   support

           0     0.9091    0.9697    0.9384       165
           1     0.2857    0.1111    0.1600        18

    accuracy                         0.8852       183
   macro avg     0.5974    0.5404    0.5492       183
weighted avg     0.8478    0.8852    0.8619       183

AUC CV (probabilidades): 0.8226


C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [8]:
# Coeficientes do modelo (interpretabilidade). As features estão escaladas dentro
# da pipeline, pelo que os coeficientes são diretamente comparáveis em magnitude.
nomes = pipe_lr_base.named_steps["prep"].get_feature_names_out()
coefs = pipe_lr_base.named_steps["modelo"].coef_[0]
coef_LR = (
    pd.DataFrame({"Variavel": nomes, "Coeficiente": coefs})
    .sort_values(by="Coeficiente", key=abs, ascending=False)
)
coef_LR.head(10)

,Variavel,Coeficiente
7,num__PM6_0,1.349998
5,num__Fle_0,1.013956
4,num__IMC,-0.752461
12,cat__Grupo_pre_2,0.650680
14,cat__Grupo_pre_4,-0.636362
13,cat__Grupo_pre_3,-0.564996
2,num__Peso,-0.557830
9,num__WR_0,0.520455
0,num__Idade,-0.506833
15,cat__Grupo_pre_5,0.443061


## 4) Teste de redundância de variáveis

Avaliamos se algumas variáveis representam a mesma informação e podem ser removidas sem perda relevante. Na EDA verificou-se forte correlação entre:

- `WT_0` e os seus componentes (`WR_0`, `WD_0`, `WAtotal_0`);
- `IMC` e as variáveis de base (`Peso`, `Altura_cm`).

Cada cenário é avaliado por **validação cruzada só no treino** e por **AUC a partir de probabilidades**, com o mesmo pré-processamento dentro da pipeline (sem data leakage).

In [9]:
# Comparação de cenários de features (Regressão Logística), por CV no treino.
cenarios = {
    "Todas as features": [],
    "Sem WT_0": ["WT_0"],
    "Sem componentes (WR_0, WD_0, WAtotal_0)": ["WR_0", "WD_0", "WAtotal_0"],
    "Sem IMC": ["IMC"],
    "Sem Peso e Altura_cm": ["Peso", "Altura_cm"],
    "Sem WT_0 e IMC": ["WT_0", "IMC"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
linhas = []
for nome, remover in cenarios.items():
    Xc = X.drop(columns=remover)
    Xc_tr = X_train.drop(columns=remover)
    pipe = prep.construir_pipeline(
        LogisticRegression(max_iter=1000, random_state=42), Xc, escalar=True
    )
    proba = cross_val_predict(pipe, Xc_tr, y_train, cv=cv, method="predict_proba")[:, 1]
    linhas.append({
        "Cenário": nome,
        "n_features": Xc_tr.shape[1],
        "AUC_CV_treino": round(roc_auc_score(y_train, proba), 4),
    })

pd.DataFrame(linhas).sort_values("AUC_CV_treino", ascending=False).reset_index(drop=True)

C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


,Cenário,n_features,AUC_CV_treino
0,Sem Peso e Altura_cm,11,0.8542
1,"Sem componentes (WR_0, WD_0, WAtotal_0)",10,0.8350
2,Sem WT_0 e IMC,11,0.8313
3,Sem IMC,12,0.8296
4,Sem WT_0,12,0.8242
5,Todas as features,13,0.8226


## 5) Afinamento por Grid Search

O scaling e a imputação já vivem dentro da pipeline (ajustados só no treino), por isso não é preciso escalar manualmente. Afinam-se os hiperparâmetros da Regressão Logística (`C`, `penalty`, `solver`, `class_weight`) por `GridSearchCV` com validação cruzada **só no treino**, otimizando a **AUC**. A grelha usa apenas combinações válidas de `solver`/`penalty` (evita fits falhados).

In [10]:
# Grid Search sobre a pipeline (CV só no treino, scoring = AUC).
pipe_lr = prep.construir_pipeline(
    LogisticRegression(max_iter=1000, random_state=42), X, escalar=True
)
param_grid = [
    {"modelo__solver": ["liblinear"], "modelo__penalty": ["l1", "l2"],
     "modelo__C": [0.1, 1, 2, 5],
     "modelo__class_weight": ["balanced", {0: 1, 1: 3}, {0: 1, 1: 10}, None]},
    {"modelo__solver": ["lbfgs"], "modelo__penalty": ["l2"],
     "modelo__C": [0.1, 1, 2, 5],
     "modelo__class_weight": ["balanced", {0: 1, 1: 3}, {0: 1, 1: 10}, None]},
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_lr = GridSearchCV(pipe_lr, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
grid_lr.fit(X_train, y_train)

pipe_lr_tuned = grid_lr.best_estimator_
print("Melhores parâmetros:", grid_lr.best_params_)
print("Melhor AUC média (CV, treino):", round(grid_lr.best_score_, 4))

Melhores parâmetros: {'modelo__C': 0.1, 'modelo__class_weight': {0: 1, 1: 10}, 'modelo__penalty': 'l1', 'modelo__solver': 'liblinear'}
Melhor AUC média (CV, treino): 0.8429


In [11]:
# Avaliação do modelo afinado no teste (probabilidades) + curva ROC + CV no treino.
print("=== Avaliação no TESTE — Regressão Logística afinada ===")
res_lr_tuned = prep.avaliar_teste(pipe_lr_tuned, X_test, y_test, titulo="LR afinada")

# Curva ROC
y_score = pipe_lr_tuned.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_score)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC (AUC = {res_lr_tuned['auc']:.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Regressão Logística afinada")
plt.legend(loc="lower right")
plt.show()

print("=== Validação cruzada (treino) — LR afinada ===")
_ = prep.validacao_cruzada_treino(pipe_lr_tuned, X_train, y_train)

=== Avaliação no TESTE — Regressão Logística afinada ===
              precision    recall  f1-score   support

           0     0.9600    0.6761    0.7934        71
           1     0.2069    0.7500    0.3243         8

    accuracy                         0.6835        79
   macro avg     0.5834    0.7130    0.5589        79
weighted avg     0.8837    0.6835    0.7459        79

AUC (probabilidades): 0.7095
=== Validação cruzada (treino) — LR afinada ===


              precision    recall  f1-score   support

           0     0.9661    0.6909    0.8057       165
           1     0.2154    0.7778    0.3373        18

    accuracy                         0.6995       183
   macro avg     0.5907    0.7343    0.5715       183
weighted avg     0.8923    0.6995    0.7596       183

AUC CV (probabilidades): 0.8219


C:\Users\edu23\Desktop\Projetos_local\Machine_Learning\preprocessing.py:167: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\edu23\AppData\Local\Temp\ipykernel_21564\4073370334.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [12]:
# Análise do threshold sobre o TREINO (para não usar o teste na escolha do corte).
y_proba_train = pipe_lr_tuned.predict_proba(X_train)[:, 1]
print("thr | precision | recall | f1")
for t in np.arange(0.1, 0.6, 0.05):
    yp = (y_proba_train >= t).astype(int)
    p = precision_score(y_train, yp, zero_division=0)
    r = recall_score(y_train, yp, zero_division=0)
    f = f1_score(y_train, yp, zero_division=0)
    print(f"{t:.2f} | {p:.2f} | {r:.2f} | {f:.2f}")

thr | precision | recall | f1
0.10 | 0.12 | 1.00 | 0.21
0.15 | 0.13 | 1.00 | 0.23
0.20 | 0.14 | 1.00 | 0.25
0.25 | 0.17 | 1.00 | 0.29
0.30 | 0.18 | 1.00 | 0.31
0.35 | 0.21 | 1.00 | 0.34
0.40 | 0.24 | 1.00 | 0.38
0.45 | 0.27 | 1.00 | 0.42
0.50 | 0.30 | 1.00 | 0.46
0.55 | 0.29 | 0.83 | 0.43


## 6) SMOTE — exemplos sintéticos da classe minoritária (feedback #4)

Testamos o **SMOTE** na Regressão Logística. Usa-se a `Pipeline` do **imbalanced-learn**, para o SMOTE ser aplicado **apenas no `fit`** (folds de treino) e nunca à validação/teste. Comparam-se as métricas **com** e **sem** SMOTE.

> Nota: o dataset é pequeno e a minoria tem poucos casos — os ganhos do SMOTE podem ser instáveis.

In [13]:
# Compara a Regressão Logística COM e SEM SMOTE, por CV no treino e no teste.
for etiqueta, usar_smote in [("SEM SMOTE", False), ("COM SMOTE", True)]:
    print("\n" + "=" * 55)
    print(f"Regressão Logística — {etiqueta}")
    print("=" * 55)
    pipe = prep.construir_pipeline(
        LogisticRegression(max_iter=1000, random_state=42), X,
        escalar=True, usar_smote=usar_smote,
    )
    pipe.fit(X_train, y_train)
    print("--- Teste ---")
    prep.avaliar_teste(pipe, X_test, y_test, titulo=f"LR {etiqueta}", plot=False)
    print("--- Validação cruzada (treino) ---")
    prep.validacao_cruzada_treino(pipe, X_train, y_train)


Regressão Logística — SEM SMOTE
--- Teste ---
              precision    recall  f1-score   support

           0     0.9167    0.9296    0.9231        71
           1     0.2857    0.2500    0.2667         8

    accuracy                         0.8608        79
   macro avg     0.6012    0.5898    0.5949        79
weighted avg     0.8528    0.8608    0.8566        79

AUC (probabilidades): 0.7165
--- Validação cruzada (treino) ---


              precision    recall  f1-score   support

           0     0.9091    0.9697    0.9384       165
           1     0.2857    0.1111    0.1600        18

    accuracy                         0.8852       183
   macro avg     0.5974    0.5404    0.5492       183
weighted avg     0.8478    0.8852    0.8619       183



C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


AUC CV (probabilidades): 0.8226

Regressão Logística — COM SMOTE


--- Teste ---
              precision    recall  f1-score   support

           0     0.9508    0.8169    0.8788        71
           1     0.2778    0.6250    0.3846         8

    accuracy                         0.7975        79
   macro avg     0.6143    0.7210    0.6317        79
weighted avg     0.8827    0.7975    0.8287        79

AUC (probabilidades): 0.7535
--- Validação cruzada (treino) ---


              precision    recall  f1-score   support

           0     0.9524    0.8485    0.8974       165
           1     0.3056    0.6111    0.4074        18

    accuracy                         0.8251       183
   macro avg     0.6290    0.7298    0.6524       183
weighted avg     0.8888    0.8251    0.8492       183

AUC CV (probabilidades): 0.7946


C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


# 7) Modelo final

Modelo final = Regressão Logística afinada por Grid Search + threshold escolhido no treino (0.45, favorecendo o *recall* da classe minoritária). Avaliação **única** no conjunto de teste.

In [14]:
# Modelo final: pipeline afinada + threshold escolhido no treino. Teste usado uma só vez.
threshold = 0.45

y_proba_final = pipe_lr_tuned.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba_final >= threshold).astype(int)

# Matriz de confusão
confusion_final = confusion_matrix(y_test, y_pred_final)
sns.heatmap(confusion_final, annot=True, fmt="d", cmap="crest")
plt.xlabel("Classe Prevista")
plt.ylabel("Classe Real")
plt.title(f"Matriz de confusão - Regressão Logística Final (threshold = {threshold})")
plt.show()

# Métricas de classificação
print("=== Avaliação no conjunto de teste (modelo final) ===")
print(classification_report(y_test, y_pred_final, digits=4))

# AUC a partir de probabilidades (independente do threshold) — feedback #2
auc_final = roc_auc_score(y_test, y_proba_final)
print("AUC:", round(auc_final, 4))

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_final)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC (AUC = {auc_final:.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Regressão Logística Final")
plt.legend(loc="lower right")
plt.show()

=== Avaliação no conjunto de teste (modelo final) ===
              precision    recall  f1-score   support

           0     0.9574    0.6338    0.7627        71
           1     0.1875    0.7500    0.3000         8

    accuracy                         0.6456        79
   macro avg     0.5725    0.6919    0.5314        79
weighted avg     0.8795    0.6456    0.7159        79

AUC: 0.7095


C:\Users\edu23\AppData\Local\Temp\ipykernel_21564\2146632028.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\edu23\AppData\Local\Temp\ipykernel_21564\2146632028.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
